# FoodLens — Food-101 Eğitim (Colab, Free T4)

Bu defter **Aşama 3** eğitim akışını çalıştırır:

1. Ortam kurulumu (torch/torchvision Colab'da hazır — **atlanır**)
2. Tam Food-101 indirir (~5.4 GB, sadece burada)
3. **Zorunlu** `validate_food101.py` ön-doğrulama — **tutmazsa eğitim BAŞLAMAZ**
4. İki aşamalı fine-tune (`train.py`) → `runs/<tarih>_<ad>/`
5. `best.pt` + `history.csv` + `summary.csv` indirilir

Beklenen sayılar: **101 sınıf, train 75.750, test 25.250, val 7.575, kalan train 68.175**.

In [ ]:
# --- 1) Ortam kurulumu (torch/torchvision Colab'da ONCEDEN gelir, AYRI KURMA) ---
import os, sys, subprocess

# proje kodu nerede? Colab'da /content/FoodLens varsayilir; REPO_URL ile clonelayabilirsiniz.
REPO_ROOT = os.environ.get('FOODLENS_ROOT', '/content/FoodLens')
REPO_URL = os.environ.get('FOODLENS_REPO_URL', '')  # ornek: https://github.com/<kullanici>/<repo>.git

if not os.path.isdir(os.path.join(REPO_ROOT, 'src')) and REPO_URL:
    print('Depo cloneleniyor:', REPO_URL)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_ROOT], check=True)

if not os.path.isdir(os.path.join(REPO_ROOT, 'src')):
    raise FileNotFoundError(
        f'FoodLens kodu bulunamadi: {REPO_ROOT}\n'
        'Defteri depo kokunde calistirin VEYA FOODLENS_ROOT / FOODLENS_REPO_URL ayarlayin.'
    )

os.chdir(REPO_ROOT)  # goreceli yollar (data/, configs/, runs/) icin onemli
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))
print('REPO_ROOT =', REPO_ROOT)

# requirements: torch/torchvision'u ATLA (Colab GPU ile eslesir), gerisini kur
skip = {'torch', 'torchvision'}
pkgs = []
for ln in open('requirements.txt', encoding='utf-8'):
    ln = ln.split('#')[0].strip()
    if not ln:
        continue
    name = ln.split('>')[0].split('=')[0].split('<')[0].strip().lower()
    if name in skip or name.startswith('uvicorn'):
        continue
    pkgs.append(ln)
print('Kurulacak paketler:', pkgs)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)

import torch, torchvision
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2) Tam Food-101 indir

Tek seferlik (~5.4 GB). torchvision arşivi `data/food-101/` altına çıkarır (`images/` + `meta/`).
Yerel makinede ASLA yapmayın — sadece Colab.

In [ ]:
from torchvision.datasets import Food101

root = os.path.join(REPO_ROOT, 'data')
os.makedirs(root, exist_ok=True)
print('Food-101 indiriliyor + cikariliyor (~5.4 GB) ...')
Food101(root=root, split='train', download=True)  # arsiv tamamini indirir (test de dahil)
print('Tamam. Icerik:')
base = os.path.join(root, 'food-101')
print('  meta/ :', sorted(os.listdir(os.path.join(base, 'meta'))))
print('  images/ sinif sayisi :', len(os.listdir(os.path.join(base, 'images'))))

## 3) Zorunlu ön-doğrulama (preflight)

`scripts/validate_food101.py` beklenen sayıları kontrol eder:
**101 sınıf · train 75.750 · test 25.250 · val 7.575 · kalan train 68.175**.

> **TUTMAZSA hata verir (`exit 1`) ve eğitim HİÇ BAŞLAMAZ.**

In [ ]:
import subprocess

rc = subprocess.call([
    sys.executable,
    os.path.join(REPO_ROOT, 'scripts', 'validate_food101.py'),
    '--config', os.path.join(REPO_ROOT, 'configs', 'default.yaml'),
])
if rc != 0:
    raise SystemExit(
        'ON-DOGRULAMA BASARISIZ — egitim BASLATILMADI. '
        'Yukaridaki farklari duzeltin, sonra tekrar calistirin.'
    )
print('On-dogrulama OK — egitim baslatilabilir.')

## 4) Eğitim (iki aşamalı fine-tune)

- **Aşama 1:** donuk backbone, sadece head (5 epoch, LR 1e-3)
- **Aşama 2:** tüm katmanlar, discriminative LR (backbone 1e-4 / head 1e-3) + cosine (12 epoch)

Çıktı: `runs/<tarih>_<ad>/` → `config.json`, `history.csv` (stage sütunu), `curves.png`, `best.pt`, `last.pt` + `runs/summary.csv`.

### Resume (Colab koparsa / süre yetmezse)
`last.pt` artık **optimizer + scheduler + epoch + history** taşır. Eğitim kesilirse:
1. Aşağıdaki **4a) Drive** hücresini çalıştır,
2. eğitim hücresinde `RESUME_DIR = '<run klasörü>'` yap,
3. hücreyi tekrar çalıştır → kaldığı **stage / epoch+1**'den devam eder.

`RESUME_DIR` boşsa yeni bir run başlar.


## 4a) Drive yedeği (resume için önerilir)

`/content` oturum kapanınca silinir. `runs/` çıktısını Drive'a yazarsan `last.pt` kalıcı olur ve yarın `--resume` ile devam edebilirsin. Çalıştırınca Google Drive erişim izni isteyen bir pencere açılır.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUNS_DIR = '/content/drive/MyDrive/FoodLens/runs'
os.makedirs(RUNS_DIR, exist_ok=True)
print('RUNS_DIR =', RUNS_DIR)


In [ ]:
from foodlens.config import load_config
from foodlens.train import build_argparser, train_run

# Colab ayarlari — gerekirse epoch / unfreeze-from-block degistirin:
EPOCHS1, EPOCHS2, UNFREEZE = 5, 12, -1   # -1 = tum bloklar

# Drive hucresi calistirildiysa runs oraya yazilir (resume icin kalici):
RUNS_DIR = globals().get('RUNS_DIR', '')

# Kaldigi yerden devam: run klasorunu (veya last.pt'yi) yaz; yeni kosu icin '' birak.
RESUME_DIR = ''   # ornek: '/content/drive/MyDrive/FoodLens/runs/20261023_..._food101_effb0'

extra = []
if RUNS_DIR:
    extra += ['--runs-dir', RUNS_DIR]
if RESUME_DIR:
    extra += ['--resume', RESUME_DIR]

cfg = load_config(os.path.join(REPO_ROOT, 'configs', 'default.yaml'))
args = build_argparser().parse_args([
    '--config', os.path.join(REPO_ROOT, 'configs', 'default.yaml'),
    '--stage1-epochs', str(EPOCHS1),
    '--stage2-epochs', str(EPOCHS2),
    '--unfreeze-from-block', str(UNFREEZE),
    '--device', 'cuda',
] + extra)

out = train_run(cfg, args)
RUN_DIR = out['run_dir']
print('Run klasoru:', RUN_DIR)
print('Resume:', out.get('resumed', False))
print('Ozet:', out['summary'])


## 5) Sonuçları indir

`best.pt` (checkpoint contract: `model_state`, `classes`, ...) → Gradio/FastAPI'de yüklenmek üzere yerelde saklanır. Drive kullanıldıysa dosyalar orada da kalır.


In [ ]:
import os
import torch

best = os.path.join(RUN_DIR, 'best.pt')
runs_root = os.path.dirname(RUN_DIR)  # runs/ (yerel veya Drive)
files_to_send = [
    best,
    os.path.join(RUN_DIR, 'history.csv'),
    os.path.join(RUN_DIR, 'curves.png'),
    os.path.join(runs_root, 'summary.csv'),
]

try:
    from google.colab import files as colab_files
    for p in files_to_send:
        print('indiriliyor:', p)
        colab_files.download(p)
except ImportError:
    print('Colab disi ortam — dosyalar yerelde duruyor:')
    for p in files_to_send:
        print('  ', p, os.path.exists(p))

print('\nEgitim tamam. best.pt contract anahtarlari:')
blob = torch.load(best, map_location='cpu', weights_only=False)
print(' ', sorted(blob.keys()), '| num_classes =', blob['num_classes'])
